In [1]:
from pathlib import Path
import json
import time
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from IPython.display import display

ROOT = Path(r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump")

CROP_FILE = ROOT / "data" / "processed" / "unified" / "unified_crop_yield_2013_2025.csv"
SOIL_FILE = ROOT / "data" / "processed" / "soil" / "district_soil_texture_nrsc_5km.csv"
CLIMATE_ANNUAL_FILE = ROOT / "data" / "processed" / "climate" / "district_climate_annual_nasa_power_2013_2025.csv"
BOUNDARY_FILE = ROOT / "data" / "raw" / "soil" / "district_boundary" / "IND_ADM2.geojson"

CLIMATE_MONTHLY_FILE = ROOT / "data" / "processed" / "climate" / "district_climate_monthly_nasa_power_2013_2025.csv"
FAILURE_FILE = ROOT / "data" / "processed" / "climate" / "nasa_power_failed_requests.csv"
CACHE_DIR = ROOT / "data" / "raw" / "climate" / "nasa_power_monthly_cache"

OUT_DIR = ROOT / "data" / "processed" / "integrated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

INTEGRATED_FILE = OUT_DIR / "crop_soil_climate_integrated_2013_2025.csv"

print("PROJECT PATHS")
for p in [CROP_FILE, SOIL_FILE, CLIMATE_ANNUAL_FILE, BOUNDARY_FILE]:
    print(f"{p.name}: {p.exists()}")

print("\nRoot:", ROOT)

PROJECT PATHS
unified_crop_yield_2013_2025.csv: True
district_soil_texture_nrsc_5km.csv: True
district_climate_annual_nasa_power_2013_2025.csv: True
IND_ADM2.geojson: True

Root: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump


In [2]:
# Load input datasets

crop = pd.read_csv(CROP_FILE)
soil = pd.read_csv(SOIL_FILE)
climate_annual = pd.read_csv(CLIMATE_ANNUAL_FILE)

print("===== INPUT DATASETS =====")
print("Crop:", crop.shape)
print("Soil:", soil.shape)
print("Climate annual:", climate_annual.shape)

print("\nCrop columns:")
print(crop.columns.tolist())

print("\nSoil columns:")
print(soil.columns.tolist())

print("\nClimate columns:")
print(climate_annual.columns.tolist())

===== INPUT DATASETS =====
Crop: (67826, 11)
Soil: (735, 6)
Climate annual: (7800, 20)

Crop columns:
['year', 'state', 'district', 'state_code', 'district_code', 'crop', 'season', 'area_ha', 'production_tonnes', 'yield_kg_ha', 'source']

Soil columns:
['district', 'clayey_fraction', 'clayey_skeletal_fraction', 'loamy_fraction', 'sandy_fraction', 'soil_type']

Climate columns:
['shapeID', 'district', 'state', 'latitude', 'longitude', 'year', 'annual_rainfall_mm', 'annual_mean_temp_c', 'annual_max_temp_c', 'annual_min_temp_c', 'annual_relative_humidity_pct', 'annual_wind_speed_m_s', 'annual_solar_radiation', 'monsoon_rainfall_mm', 'monsoon_mean_temp_c', 'monsoon_max_temp_c', 'monsoon_min_temp_c', 'monsoon_relative_humidity_pct', 'monsoon_wind_speed_m_s', 'monsoon_solar_radiation']


In [3]:
# Clean text keys without changing the original row count

def clean_key(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

for df in [crop, soil, climate_annual]:
    for col in ["state", "district"]:
        if col in df.columns:
            df[col] = clean_key(df[col])

print("Text keys cleaned.")

Text keys cleaned.


In [4]:
# FINAL ROBUST YEAR PARSING
# This handles numeric values, strings, ranges, slash formats, and common agricultural-year representations.

def parse_crop_year(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()

    # Find the first 4-digit year anywhere in the value.
    match = re.search(r"(?:19|20)\d{2}", s)

    if match:
        year = int(match.group(0))
        if 1900 <= year <= 2100:
            return year

    return pd.NA


crop_year_original = crop["year"].copy()

crop["year"] = crop_year_original.map(parse_crop_year).astype("Int64")

climate_annual["year"] = pd.to_numeric(
    climate_annual["year"],
    errors="coerce"
).astype("Int64")

EXPECTED_CROP_YEARS = set(range(2013, 2025))

actual_crop_years = set(
    crop["year"].dropna().astype(int).unique()
)

climate_years = set(
    climate_annual["year"].dropna().astype(int).unique()
)

print("=" * 65)
print("YEAR VALIDATION")
print("=" * 65)

print("\nCrop rows:", len(crop))
print("Crop missing years:", crop["year"].isna().sum())
print("Crop year coverage:", sorted(actual_crop_years))
print("Expected crop years:", sorted(EXPECTED_CROP_YEARS))

print("\nClimate year coverage:", sorted(climate_years))

missing_crop_years = EXPECTED_CROP_YEARS - actual_crop_years
extra_crop_years = actual_crop_years - EXPECTED_CROP_YEARS

print("\nMissing crop years:", sorted(missing_crop_years))
print("Extra crop years:", sorted(extra_crop_years))

if crop["year"].isna().any():
    bad_values = crop_year_original[crop["year"].isna()].drop_duplicates().head(20).tolist()
    raise ValueError(
        "Some crop year values could not be parsed. "
        f"Examples: {bad_values}"
    )

if actual_crop_years != EXPECTED_CROP_YEARS:
    raise ValueError(
        "Crop year coverage is incorrect.\n"
        f"Expected: {sorted(EXPECTED_CROP_YEARS)}\n"
        f"Found:    {sorted(actual_crop_years)}"
    )

missing_climate_years = EXPECTED_CROP_YEARS - climate_years

if missing_climate_years:
    raise ValueError(
        "Climate data is missing crop years: "
        f"{sorted(missing_climate_years)}"
    )

print("\nYEAR VALIDATION PASSED")
print("Crop years represent agricultural years 2013-14 through 2024-25.")
print("Climate may additionally contain calendar year 2025.")

YEAR VALIDATION

Crop rows: 67826
Crop missing years: 0
Crop year coverage: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Expected crop years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Climate year coverage: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Missing crop years: []
Extra crop years: []

YEAR VALIDATION PASSED
Crop years represent agricultural years 2013-14 through 2024-25.
Climate may additionally contain calendar year 2025.


In [5]:
#Recover missing NASA POWER climate requests
API_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

PARAMETERS = [
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "RH2M",
    "WS10M",
    "PRECTOTCORR",
    "PRECTOTCORR_SUM",
    "ALLSKY_SFC_SW_DWN"
]

START_YEAR = 2013
END_YEAR = 2025

CACHE_DIR.mkdir(parents=True, exist_ok=True)

if FAILURE_FILE.exists():
    failed = pd.read_csv(FAILURE_FILE)
else:
    failed = pd.DataFrame()

print("NASA cache directory:", CACHE_DIR)
print("Cached JSON files:", len(list(CACHE_DIR.glob("*.json"))))
print("Recorded failed requests:", len(failed))

NASA cache directory: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\raw\climate\nasa_power_monthly_cache
Cached JSON files: 735
Recorded failed requests: 20


In [6]:
# Retry failed NASA requests safely.
# Important:mIf a failed request already has a valid cache file, it is NOT requested again.

def cache_file_for_shape(shape_id):
    safe_id = str(shape_id).replace("/", "_").replace("\\", "_")
    return CACHE_DIR / f"{safe_id}.json"


def valid_nasa_cache(path):
    if not path.exists():
        return False

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        params = data.get("properties", {}).get("parameter", {})
        return isinstance(params, dict) and len(params) > 0

    except Exception:
        return False


def retry_nasa_request(row, max_attempts=4):
    cache_path = cache_file_for_shape(row["shapeID"])

    if valid_nasa_cache(cache_path):
        return True, "already_cached"

    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": float(row["longitude"]),
        "latitude": float(row["latitude"]),
        "start": START_YEAR,
        "end": END_YEAR,
        "format": "JSON",
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=90
            )

            if response.status_code == 429:
                wait_seconds = min(60 * attempt, 180)
                print(
                    f"429 for {row.get('district', row['shapeID'])}; "
                    f"waiting {wait_seconds}s..."
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

            data = response.json()

            if "properties" not in data:
                raise ValueError("NASA response has no 'properties'.")

            if "parameter" not in data["properties"]:
                raise ValueError("NASA response has no 'parameter'.")

            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(data, f)

            return True, "recovered"

        except Exception as exc:
            last_error = repr(exc)

            if attempt < max_attempts:
                time.sleep(5 * attempt)

    return False, last_error


remaining_failures = []

if not failed.empty:
    required_cols = {"shapeID", "latitude", "longitude"}

    if required_cols.issubset(failed.columns):
        for i, row in failed.iterrows():
            ok, status = retry_nasa_request(row)

            if ok:
                print(
                    f"[{i+1}/{len(failed)}] "
                    f"{row.get('district', row['shapeID'])}: {status}"
                )
            else:
                print(
                    f"[{i+1}/{len(failed)}] "
                    f"{row.get('district', row['shapeID'])}: STILL FAILED"
                )
                remaining_failures.append({
                    "shapeID": row["shapeID"],
                    "district": row.get("district", pd.NA),
                    "state": row.get("state", pd.NA),
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "error": status
                })

            time.sleep(2)

    else:
        print("Failure file does not contain the required coordinate columns.")
else:
    print("No recorded NASA failures to retry.")

remaining_failures = pd.DataFrame(remaining_failures)

print("\nRemaining NASA failures:", len(remaining_failures))
print("Total cached JSON files:", len(list(CACHE_DIR.glob("*.json"))))

[1/20] Alipurduar: already_cached
[2/20] Jhargram: already_cached
[3/20] Paschim Barddhaman: already_cached
[4/20] Kalimpong: already_cached
[5/20] Narayanpet: already_cached
[6/20] Mulugu: already_cached
[7/20] Niwari: already_cached
[8/20] Pakke Kessang: already_cached
[9/20] Kamle: already_cached
[10/20] Shi Yomi: already_cached
[11/20] Lower Siang: already_cached
[12/20] Leparada: already_cached
[13/20] Hnahthial: already_cached
[14/20] Saitual: already_cached
[15/20] Khawzawl: already_cached
[16/20] Tirupathur: already_cached
[17/20] Ranipet: already_cached
[18/20] Chengalputtu: already_cached
[19/20] Mayiladuthurai: already_cached
[20/20] Kallakurichi: already_cached

Remaining NASA failures: 0
Total cached JSON files: 735


In [7]:
#Rebuild climate from the cache
boundary = gpd.read_file(BOUNDARY_FILE)

boundary["district"] = clean_key(boundary["shapeName"])
boundary["shapeID"] = boundary["shapeID"].astype(str)

print("Boundary shape:", boundary.shape)
print("Boundary CRS:", boundary.crs)
print("Boundary districts:", boundary["district"].nunique())

Boundary shape: (735, 7)
Boundary CRS: EPSG:4326
Boundary districts: 728


In [8]:
# Build district -> state mapping from the crop master.
# A district is assigned a state only when the crop master associates it with exactly one state.

crop_state_pairs = (
    crop[["state", "district"]]
    .dropna()
    .drop_duplicates()
)

district_state_map = (
    crop_state_pairs
    .groupby("district")["state"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

def unique_state_for_district(district):
    states = district_state_map.get(str(district), [])
    if len(states) == 1:
        return states[0]
    return pd.NA

print("Districts with unique state mapping:",
      sum(len(v) == 1 for v in district_state_map.values()))
print("Ambiguous district names:",
      sum(len(v) > 1 for v in district_state_map.values()))

Districts with unique state mapping: 800
Ambiguous district names: 3


In [9]:
# Rebuild monthly climate records from every valid NASA cache file.

records = []

for json_file in CACHE_DIR.glob("*.json"):
    try:
        shape_id = json_file.stem

        match = boundary.loc[
            boundary["shapeID"] == shape_id
        ]

        if match.empty:
            continue

        district = match.iloc[0]["district"]
        state = unique_state_for_district(district)

        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        params = (
            data
            .get("properties", {})
            .get("parameter", {})
        )

        if not isinstance(params, dict):
            continue

        all_months = set()

        for values in params.values():
            if isinstance(values, dict):
                all_months.update(str(k) for k in values.keys())

        for ym in sorted(all_months):
            if not re.fullmatch(r"\d{6}", ym):
                continue

            year = int(ym[:4])
            month = int(ym[4:])

            if not (START_YEAR <= year <= END_YEAR):
                continue

            row = {
                "shapeID": shape_id,
                "state": state,
                "district": district,
                "year": year,
                "month": month,
            }

            for parameter_name in PARAMETERS:
                row[parameter_name] = (
                    params
                    .get(parameter_name, {})
                    .get(ym, np.nan)
                )

            records.append(row)

    except Exception as exc:
        print("Skipped:", json_file.name, "|", repr(exc))


monthly_rebuilt = pd.DataFrame(records)

if monthly_rebuilt.empty:
    raise ValueError(
        "No climate records could be rebuilt from the NASA cache."
    )

for col in PARAMETERS:
    monthly_rebuilt[col] = pd.to_numeric(
        monthly_rebuilt[col],
        errors="coerce"
    )

print("Rebuilt monthly rows:", len(monthly_rebuilt))
print("Unique districts:", monthly_rebuilt["district"].nunique())
print("Years:", sorted(monthly_rebuilt["year"].unique()))

Rebuilt monthly rows: 124215
Unique districts: 728
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [10]:
# Create annual climate features.

annual_climate = (
    monthly_rebuilt
    .groupby(
        ["state", "district", "year"],
        dropna=False,
        as_index=False
    )
    .agg(
        annual_rainfall_mm=("PRECTOTCORR_SUM", "sum"),
        annual_mean_temp_c=("T2M", "mean"),
        annual_max_temp_c=("T2M_MAX", "mean"),
        annual_min_temp_c=("T2M_MIN", "mean"),
        annual_relative_humidity_pct=("RH2M", "mean"),
        annual_wind_speed_m_s=("WS10M", "mean"),
        annual_solar_radiation=("ALLSKY_SFC_SW_DWN", "mean"),
    )
)

# Monsoon = June, July, August, September.
monsoon = monthly_rebuilt[
    monthly_rebuilt["month"].isin([6, 7, 8, 9])
].copy()

monsoon_climate = (
    monsoon
    .groupby(
        ["state", "district", "year"],
        dropna=False,
        as_index=False
    )
    .agg(
        monsoon_rainfall_mm=("PRECTOTCORR_SUM", "sum"),
        monsoon_mean_temp_c=("T2M", "mean"),
        monsoon_max_temp_c=("T2M_MAX", "mean"),
        monsoon_min_temp_c=("T2M_MIN", "mean"),
        monsoon_relative_humidity_pct=("RH2M", "mean"),
        monsoon_wind_speed_m_s=("WS10M", "mean"),
        monsoon_solar_radiation=("ALLSKY_SFC_SW_DWN", "mean"),
    )
)

climate_unique = annual_climate.merge(
    monsoon_climate,
    on=["state", "district", "year"],
    how="left",
    validate="one_to_one"
)

print("Annual climate rows:", len(climate_unique))
print("Climate years:", sorted(climate_unique["year"].unique()))

print(
    "Unique state-district-year keys:",
    climate_unique[["state", "district", "year"]].drop_duplicates().shape[0]
)

Annual climate rows: 9464
Climate years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Unique state-district-year keys: 9464


In [11]:
#Build safe soil lookup
soil_keyed = soil.copy()

soil_keyed["district"] = clean_key(soil_keyed["district"])

# Use crop data to assign state.
crop_district_states = (
    crop[["state", "district"]]
    .dropna()
    .drop_duplicates()
    .groupby("district")["state"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

def soil_state_lookup(district):
    states = crop_district_states.get(str(district), [])
    return states[0] if len(states) == 1 else pd.NA

soil_keyed["state"] = soil_keyed["district"].map(
    soil_state_lookup
)

soil_unique = soil_keyed[
    soil_keyed["state"].notna()
].copy()

# The soil table can contain duplicate polygons for one state-district.
# Keep exactly one lookup record per merge key.
soil_unique = soil_unique.drop_duplicates(
    subset=["state", "district"],
    keep="first"
)

print("SOIL LOOKUP")
print("Raw soil rows:", len(soil_keyed))
print("Usable soil rows:", len(soil_unique))
print("Unresolved soil rows:",
      soil_keyed["state"].isna().sum())

print(
    "Unique soil keys:",
    soil_unique[["state", "district"]].drop_duplicates().shape[0]
)

SOIL LOOKUP
Raw soil rows: 735
Usable soil rows: 614
Unresolved soil rows: 117
Unique soil keys: 614


In [12]:
#Build safe climate lookup
climate_keyed = climate_unique.copy()

climate_keyed["state"] = clean_key(climate_keyed["state"])
climate_keyed["district"] = clean_key(climate_keyed["district"])

climate_keyed["year"] = pd.to_numeric(
    climate_keyed["year"],
    errors="coerce"
).astype("Int64")

climate_keyed = climate_keyed[
    climate_keyed["state"].notna()
    & climate_keyed["district"].notna()
    & climate_keyed["year"].notna()
].copy()

# Ensure one row per state-district-year.
duplicate_mask = climate_keyed.duplicated(
    subset=["state", "district", "year"],
    keep=False
)

print(" CLIMATE LOOKUP")
print("Usable climate rows:", len(climate_keyed))
print("Duplicate keys:", duplicate_mask.sum())

if duplicate_mask.any():
    numeric_cols = [
        c for c in climate_keyed.columns
        if c not in ["state", "district", "year"]
        and pd.api.types.is_numeric_dtype(climate_keyed[c])
    ]

    climate_keyed = (
        climate_keyed
        .groupby(
            ["state", "district", "year"],
            as_index=False
        )[numeric_cols]
        .mean()
    )

print(
    "Final unique climate keys:",
    climate_keyed[
        ["state", "district", "year"]
    ].drop_duplicates().shape[0]
)

 CLIMATE LOOKUP
Usable climate rows: 7982
Duplicate keys: 0
Final unique climate keys: 7982


In [13]:
#Merge soil and climate into the crop master

# Preserve an ID for row-count validation.
crop_integrated = crop.copy()
crop_integrated["_original_crop_row_id"] = np.arange(
    len(crop_integrated)
)

SOIL_COLUMNS = [
    "state",
    "district",
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction",
    "soil_type",
]

available_soil_columns = [
    c for c in SOIL_COLUMNS
    if c in soil_unique.columns
]

required_soil_columns = [
    "state",
    "district"
]

if not all(c in available_soil_columns for c in required_soil_columns):
    raise ValueError("Soil lookup does not contain state/district keys.")

crop_integrated = crop_integrated.merge(
    soil_unique[available_soil_columns],
    on=["state", "district"],
    how="left",
    validate="many_to_one",
    indicator="_soil_merge"
)

print("After soil merge:", crop_integrated.shape)

# Climate features.
CLIMATE_COLUMNS = [
    "state",
    "district",
    "year",
    "annual_rainfall_mm",
    "annual_mean_temp_c",
    "annual_max_temp_c",
    "annual_min_temp_c",
    "annual_relative_humidity_pct",
    "annual_wind_speed_m_s",
    "annual_solar_radiation",
    "monsoon_rainfall_mm",
    "monsoon_mean_temp_c",
    "monsoon_max_temp_c",
    "monsoon_min_temp_c",
    "monsoon_relative_humidity_pct",
    "monsoon_wind_speed_m_s",
    "monsoon_solar_radiation",
]

available_climate_columns = [
    c for c in CLIMATE_COLUMNS
    if c in climate_keyed.columns
]

if not all(
    c in available_climate_columns
    for c in ["state", "district", "year"]
):
    raise ValueError(
        "Climate lookup does not contain state/district/year keys."
    )

crop_integrated = crop_integrated.merge(
    climate_keyed[available_climate_columns],
    on=["state", "district", "year"],
    how="left",
    validate="many_to_one",
    indicator="_climate_merge"
)

print("After climate merge:", crop_integrated.shape)

After soil merge: (67826, 18)
After climate merge: (67826, 33)


In [14]:
#Integration validation
original_rows = len(crop)
integrated_rows = len(crop_integrated)

duplicate_original_ids = (
    crop_integrated["_original_crop_row_id"]
    .duplicated()
    .sum()
)

soil_matched = (
    crop_integrated["_soil_merge"] == "both"
).sum()

climate_matched = (
    crop_integrated["_climate_merge"] == "both"
).sum()

print("=" * 70)
print("INTEGRATION VALIDATION")
print("=" * 70)

print("Original crop rows:", original_rows)
print("Integrated rows:", integrated_rows)
print("Duplicate original crop IDs:", duplicate_original_ids)

print("\nSoil matched rows:", soil_matched)
print("Soil unmatched rows:", original_rows - soil_matched)

print("\nClimate matched rows:", climate_matched)
print("Climate unmatched rows:", original_rows - climate_matched)

if integrated_rows != original_rows:
    raise ValueError(
        "ERROR: Integration changed the original crop row count."
    )

if duplicate_original_ids != 0:
    raise ValueError(
        "ERROR: Integration duplicated original crop rows."
    )

print("\nROW-COUNT VALIDATION PASSED")

INTEGRATION VALIDATION
Original crop rows: 67826
Integrated rows: 67826
Duplicate original crop IDs: 0

Soil matched rows: 55696
Soil unmatched rows: 12130

Climate matched rows: 55696
Climate unmatched rows: 12130

ROW-COUNT VALIDATION PASSED


In [15]:
# Check crop-year coverage after integration.

final_crop_years = set(
    crop_integrated["year"]
    .dropna()
    .astype(int)
    .unique()
)

print("Final crop year coverage:", sorted(final_crop_years))

if final_crop_years != EXPECTED_CROP_YEARS:
    raise ValueError(
        "Final integrated crop-year coverage is incorrect."
    )

print("Final crop-year validation PASSED.")

Final crop year coverage: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Final crop-year validation PASSED.


In [16]:
print("ENVIRONMENTAL MERGE AUDIT")

soil_unmatched = crop_integrated[
    crop_integrated["_soil_merge"] != "both"
][["state", "district"]].drop_duplicates()

climate_unmatched = crop_integrated[
    crop_integrated["_climate_merge"] != "both"
][["state", "district", "year"]].drop_duplicates()

print("\nUnique unmatched soil keys:", len(soil_unmatched))
if not soil_unmatched.empty:
    display(soil_unmatched.head(20))

print("\nUnique unmatched climate keys:", len(climate_unmatched))
if not climate_unmatched.empty:
    display(climate_unmatched.head(20))

print("\nThis audit is informational; unmatched environmental values are not silently filled.")

ENVIRONMENTAL MERGE AUDIT

Unique unmatched soil keys: 192


,state,district
4,Andaman And Nicobar Islands,North And Middle Andaman
10,Andaman And Nicobar Islands,South Andamans
16,Andhra Pradesh,Ananthapuramu
138,Andhra Pradesh,Y.S.R. Kadapa
303,Assam,Kamrup Metro
310,Assam,Karbi Anglong
331,Assam,Marigaon
366,Assam,Sribhumi
598,Bihar,Purbi Champaran
696,Chhattisgarh,Balodabazar-Bhatapara



Unique unmatched climate keys: 1526


,state,district,year
4,Andaman And Nicobar Islands,North And Middle Andaman,2013
10,Andaman And Nicobar Islands,South Andamans,2013
16,Andhra Pradesh,Ananthapuramu,2013
138,Andhra Pradesh,Y.S.R. Kadapa,2013
303,Assam,Kamrup Metro,2013
310,Assam,Karbi Anglong,2013
331,Assam,Marigaon,2013
366,Assam,Sribhumi,2013
598,Bihar,Purbi Champaran,2013
696,Chhattisgarh,Balodabazar-Bhatapara,2013



This audit is informational; unmatched environmental values are not silently filled.


In [17]:
#Missing value audit
environmental_columns = [
    c for c in crop_integrated.columns
    if (
        c.startswith("annual_")
        or c.startswith("monsoon_")
        or c.endswith("_fraction")
        or c == "soil_type"
    )
]

missing_summary = (
    crop_integrated[environmental_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(crop_integrated)
    * 100
)

display(missing_summary)

print(
    "\nColumns with missing environmental values:",
    (missing_summary["missing_count"] > 0).sum()
)

,missing_count,missing_pct
clayey_fraction,12130,17.883997
clayey_skeletal_fraction,12130,17.883997
loamy_fraction,12130,17.883997
sandy_fraction,12130,17.883997
soil_type,12130,17.883997
annual_rainfall_mm,12130,17.883997
annual_mean_temp_c,12130,17.883997
annual_max_temp_c,12130,17.883997
annual_min_temp_c,12130,17.883997
annual_relative_humidity_pct,12130,17.883997



Columns with missing environmental values: 19


In [18]:
# Drop temporary validation columns.

temporary_columns = [
    "_original_crop_row_id",
    "_soil_merge",
    "_climate_merge",
]

final_integrated = crop_integrated.drop(
    columns=[
        c for c in temporary_columns
        if c in crop_integrated.columns
    ]
)

# Reorder year near the front if possible.
preferred_order = [
    "year",
    "state",
    "district",
    "state_code",
    "district_code",
    "crop",
    "season",
    "area_ha",
    "production_tonnes",
    "yield_kg_ha",
    "source",
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction",
    "soil_type",
    "annual_rainfall_mm",
    "annual_mean_temp_c",
    "annual_max_temp_c",
    "annual_min_temp_c",
    "annual_relative_humidity_pct",
    "annual_wind_speed_m_s",
    "annual_solar_radiation",
    "monsoon_rainfall_mm",
    "monsoon_mean_temp_c",
    "monsoon_max_temp_c",
    "monsoon_min_temp_c",
    "monsoon_relative_humidity_pct",
    "monsoon_wind_speed_m_s",
    "monsoon_solar_radiation",
]

ordered_columns = [
    c for c in preferred_order
    if c in final_integrated.columns
]

remaining_columns = [
    c for c in final_integrated.columns
    if c not in ordered_columns
]

final_integrated = final_integrated[
    ordered_columns + remaining_columns
]

print("Final integrated shape:", final_integrated.shape)
print("Final columns:", len(final_integrated.columns))

Final integrated shape: (67826, 30)
Final columns: 30


In [19]:
# Save final integrated dataset.

final_integrated.to_csv(
    INTEGRATED_FILE,
    index=False
)

print("=" * 70)
print("FINAL DATASET SAVED")
print("=" * 70)

print("File:", INTEGRATED_FILE)
print("Rows:", len(final_integrated))
print("Columns:", len(final_integrated.columns))
print("Size (MB):", round(INTEGRATED_FILE.stat().st_size / (1024**2), 2))

print("\nFirst 5 rows:")
display(final_integrated.head())

FINAL DATASET SAVED
File: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_integrated_2013_2025.csv
Rows: 67826
Columns: 30
Size (MB): 17.35

First 5 rows:


,year,state,district,state_code,district_code,crop,season,area_ha,production_tonnes,yield_kg_ha,...,annual_relative_humidity_pct,annual_wind_speed_m_s,annual_solar_radiation,monsoon_rainfall_mm,monsoon_mean_temp_c,monsoon_max_temp_c,monsoon_min_temp_c,monsoon_relative_humidity_pct,monsoon_wind_speed_m_s,monsoon_solar_radiation
0,2013,Andaman And Nicobar Islands,Nicobars,35,603,Maize,Annual,9.70,5.1,526.0,...,80.76,5.333846,16.925385,1010.69,27.7575,28.69,26.475,82.7975,7.02,15.765
1,2013,Andaman And Nicobar Islands,Nicobars,35,603,Rice,Annual,2.65,6.3,2377.0,...,80.76,5.333846,16.925385,1010.69,27.7575,28.69,26.475,82.7975,7.02,15.765
2,2013,Andaman And Nicobar Islands,Nicobars,35,603,Sugarcane,Annual,11.00,373.4,33945.0,...,80.76,5.333846,16.925385,1010.69,27.7575,28.69,26.475,82.7975,7.02,15.765
3,2013,Andaman And Nicobar Islands,Nicobars,35,603,Urad,Annual,6.30,2.5,397.0,...,80.76,5.333846,16.925385,1010.69,27.7575,28.69,26.475,82.7975,7.02,15.765
4,2013,Andaman And Nicobar Islands,North And Middle Andaman,35,632,Arhar/Tur,Annual,1.00,3.0,3000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Final hard checks.

assert len(final_integrated) == len(crop), (
    "Final dataset row count does not match crop master."
)

assert final_integrated["year"].notna().all(), (
    "Final dataset contains missing crop years."
)

assert set(
    final_integrated["year"].astype(int).unique()
) == EXPECTED_CROP_YEARS, (
    "Final dataset crop-year coverage is incorrect."
)

assert INTEGRATED_FILE.exists(), (
    "Final integrated file was not created."
)

print("=" * 70)
print("ALL FINAL CHECKS PASSED")
print("=" * 70)
print("Rows:", len(final_integrated))
print("Crop years:", sorted(EXPECTED_CROP_YEARS))
print("Output:", INTEGRATED_FILE)

ALL FINAL CHECKS PASSED
Rows: 67826
Crop years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Output: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_integrated_2013_2025.csv
